In [1]:
# input
import pandas as pd
import argparse
import numpy as np

In [13]:
# load TE
TE = pd.read_csv("/home/tilman/termites/repeatmasker/Mbel/hifiasm_scaff10x_arks.fa_modifiedheader.out", comment='#', sep=' ', header=None, skipinitialspace=True)
TE_header = 'SW_score perc_div perc_del perc_ins query_seq query_seq_begin query_seq_end query_seq_left comp/cons matching_repeat repeat_class_family repeat_seq_begin repeat_seq_end repeat_seq_left ID higher_score'.split(' ')
TE.columns = TE_header
TE['class'] = [ i.split('/')[0] for i in TE["repeat_class_family"]]
TE['family'] = [ i.split('/')[-1] for i in TE["repeat_class_family"]]
TE['scaffold']=TE['query_seq']

In [10]:
def get_frac_per_win(TE, TE_type="all", winsize=10_000):
    if not TE_type=="all":
        TE=TE.loc[TE["class"]==TE_type]
    win_dict = {}
    win_size = winsize
    for i, te in TE.iterrows():
        
        # check bin TE is starting in:
        bin_id = np.floor((te.query_seq_begin/win_size))
        bin_bottom= bin_id*win_size
        bin_top = (bin_id+1)*win_size
        win_dict.setdefault(te.scaffold, {}).setdefault(bin_id, {"counter":0,'id':[], "type":[],"frac":0, "lens":[] })
        win_dict[te.scaffold][bin_id]["counter"] +=1
        win_dict[te.scaffold][bin_id]["id"].append(te.matching_repeat)
        win_dict[te.scaffold][bin_id]["type"].append(te['class'])
        frac = (np.min([bin_top,te.query_seq_end]) - np.max([bin_bottom, te.query_seq_begin]))
        win_dict[te.scaffold][bin_id]["lens"].append(frac)
        win_dict[te.scaffold][bin_id]["frac"] += frac
        while te.query_seq_end>(bin_top*win_size):
            bin_id = bin_id+1
            bin_bottom= bin_id*win_size
            bin_top = (bin_id+1)*win_size
            win_dict.setdefault(bin_id, {"counter":0,'id':[], "type":[],frac:0 })
            win_dict[te.scaffold][bin_id]["counter"] +=1
            win_dict[te.scaffold][bin_id]["id"].append(te.matching_repeat)
            win_dict[te.scaffold][bin_id]["type"].append(te['class'])
            frac = (np.min([bin_top,te.query_seq_end]) - np.max([bin_bottom, te.query_seq_begin]))
            win_dict[te.scaffold][bin_id]["lens"].append(frac)
            win_dict[te.scaffold][bin_id]["frac"] += frac
            
 
    frac_df_all = []
    for scaffold, item in win_dict.items():
        for bin_name, values in item.items():
            line = [scaffold, bin_name*win_size, (bin_name+1)*win_size, values['frac']]
            frac_df_all.append(line)
    frac_df_all = pd.DataFrame(frac_df_all)
    frac_df_all.columns = ["scaffold", 'start_bin', "stop_win", "frac_rep"]
    
    return win_dict, frac_df_all

In [2]:
def get_peaks(data,bin_size_kb=2, comp_flanksize_kb=50, fold_thresh=5):
    scaff_l = list(set(data.Scaffold))
    peak = []
    nopeak = []
    
    for scaff in scaff_l:
        onekbfile = data.loc[data.Scaffold==scaff].reset_index()
        
        for i, bin in onekbfile.iterrows():
            #print(i)
            #print(scaff)
            #print(bin.Start_bp)
            #print(onekbfile)
            #print(onekbfile.iloc[i:i+bin_size_kb-1])
            peak_mean_rho = onekbfile.iloc[i:i+bin_size_kb-1].Rho_kb.mean()
            #print(peak_mean_rho)

            if i == onekbfile.index.max():
                peak_end = onekbfile.iloc[i,].End_bp
            else:
                try:
                    peak_end = onekbfile.iloc[i+1,].End_bp
                except IndexError:
                    #print(onekbfile.index.max())
                    #print(i)
                    #print(bin)
                    peak_end = onekbfile.iloc[i,].End_bp
                    raise IndexError
                    
                    
            flank_start = i-50
            if flank_start<0: # make sure we dont overshoot the scaffold boundary
                flank_start=0
    
            flank_stop = i+bin_size_kb-1+50
            if flank_stop>onekbfile.index.max(): # make sure we dont overshoot the scaffold boundary
                flank_stop=onekbfile.index.max()

            flank_mean_rho = onekbfile.iloc[flank_start:flank_stop].Rho_kb.mean()
            #print(scaff)
            #print(bin.Start_bp)
            #print(peak_end)
            #print(peak_mean_rho)
            #print(flank_mean_rho)
            #print(peak_mean_rho/flank_mean_rho)
            fold_diff = (peak_mean_rho/flank_mean_rho)
            if fold_diff>fold_thresh:
                #print("PEAK")
                peak.append([scaff, bin.Start_bp, peak_end, True, fold_diff, peak_mean_rho, flank_mean_rho])
            else:
                nopeak.append([scaff, bin.Start_bp, peak_end, False, fold_diff, peak_mean_rho, flank_mean_rho])
        #    else: 
        #        peak.append('False')
    return peak, nopeak
        
    

In [14]:
win_dict, frac_df = get_frac_per_win(TE=TE, TE_type="all", winsize=1_000)

In [15]:
data = pd.read_csv("../data/concat_Mbel_excl5scaff_rmind_hardfilt_exhet_biall_dp_qfilt_mac2_maxmiss06_phimp_LDhat_bpen1_statres_100N_filtMQ70depth2stdev.txt_w1kb", sep='\t')

In [4]:
p, pn = get_peaks(data=data)

In [5]:
peaks = pd.DataFrame(p)
nopeaks = pd.DataFrame(pn)

In [6]:
peaks.columns = ["scaffold", "bin_Start_bp", "bin_Stop_bp", "peak", "fold_diff", "peak_mean_rho", "flank_mean_rho"]
nopeaks.columns = ["scaffold", "bin_Start_bp", "bin_Stop_bp", "no_peak", "fold_diff", "peak_mean_rho", "flank_mean_rho"]

In [7]:
peaks.bin_Start_bp = peaks.bin_Start_bp.astype(int)#.astype(str)
peaks.bin_Stop_bp = peaks.bin_Stop_bp.astype(int)#.astype(str)
nopeaks.bin_Start_bp = nopeaks.bin_Start_bp.astype(int)#.astype(str)
nopeaks.bin_Stop_bp = nopeaks.bin_Stop_bp.astype(int)#.astype(str)

In [17]:
frac_rho = frac_df.merge(data, left_on=['scaffold', 'start_bin'], right_on=['Scaffold', 'Start_bp'], how='outer')

In [8]:
data = pd.read_csv("../data/concat_Mbel_excl5scaff_rmind_hardfilt_exhet_biall_dp_qfilt_mac2_maxmiss06_phimp_LDhat_bpen1_statres_100N_filtMQ70depth2stdev.txt_w1kb", sep='\t')

In [ ]:
fr_list = []
for i, k in peaks.iterrows():
    scaffold = k.scaffold
    start_bin = k.bin_Start_bp
    stop_bin = k.bin_Stop_bp
    fr = frac_rho.loc[frac_rho.Scaffold==scaffold].loc[frac_rho.Start_bp>=start_bin].loc[frac_rho.End_bp<=stop_bin].frac_rep.sum()
    fr_list.append(fr)

In [ ]:
fr_list_np = []
for i, k in nopeaks.iterrows():
    scaffold = k.scaffold
    start_bin = k.bin_Start_bp
    stop_bin = k.bin_Stop_bp
    fr = frac_rho.loc[frac_rho.Scaffold==scaffold].loc[frac_rho.Start_bp>=start_bin].loc[frac_rho.End_bp<=stop_bin].frac_rep.sum()
    fr_list_np.append(fr)

In [ ]:
nopeaks["frac_rep"] = fr_list_np

In [ ]:
peaks["frac_rep"] = fr_list

In [ ]:
peaks.to_csv('peak_intermediate.csv')

In [ ]:
nopeaks.to_csv('nopeak_intermediate.csv')